# Chapter 5 — Debate Pool

## 학습 목표
1. **4가지 답변 전략** (single / self-reflect / multi-LLM debate / hybrid) 을 정확도-비용 트레이드오프로 비교.
2. **4 provider 동시 debate** (Kimi/DeepSeek/OpenAI/Grok) 라운드 + moderator 합성.
3. Opik trace로 reasoning step 수 vs 응답 품질을 산점도화.

## 전제 조건
- Ch 4의 citation envelope 패턴 이해
- 최소 2 provider 키 설정 (1개로는 debate 데모 불가)

In [ ]:
from pathlib import Path
import os, sys, json, time, asyncio
from dotenv import load_dotenv

HERE = Path.cwd()
if HERE.name != 'teaching-resource' and (HERE / 'teaching-resource').exists():
    os.chdir(HERE / 'teaching-resource')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
for cand in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if cand.exists():
        load_dotenv(cand, override=False); break

from _shared.opik_setup import setup_opik, opik_console_link, teardown_opik
from _shared.providers import compare_providers, available_providers, chat, PROVIDERS
from _shared.finder_loader import sample_per_category

trace_info = setup_opik('05', only_jsonl=not os.getenv('OPIK_API_KEY'))
print('Opik project:', trace_info['project'])

CONFIGURED = [n for n, ok in available_providers().items() if ok]
print('configured providers:', CONFIGURED)
if len(CONFIGURED) < 2:
    print('⚠️ debate 데모는 최소 2개 provider 가 필요합니다.')

## 5.0 평가셋 — FinDER 카테고리당 1건씩

In [ ]:
EVAL = sample_per_category(1)  # 8 categories × 1 = 8 records
for r in EVAL[:4]:
    print(f"[{r['category']:18s}] {r['question'][:80]}")

## 5.1 Strategy A — Single LLM (Baseline)

1개 모델 × 1회 호출. 측정 항목: 정확도(휴리스틱) / 환각률 / 토큰비용 / latency.

In [ ]:
ANSWER_SYS = (
    'Answer the financial question concisely in Korean using ONLY the provided context. '
    'If context is insufficient, reply "근거 없음". '
    'Cite source ids inline as [src:id].'
)

def build_prompt(rec: dict) -> tuple[str, str]:
    ctx = f"[src:{rec.get('id')}] {rec.get('text', '')[:1800]}"
    user = f"Context:\n{ctx}\n\nQuestion: {rec.get('question')}"
    return ANSWER_SYS, user

def single(rec: dict, provider: str = 'openai') -> dict:
    sys_p, user = build_prompt(rec)
    t0 = time.perf_counter()
    resp = chat(provider, user=user, system=sys_p, max_tokens=300, temperature=0.0)
    return {
        'strategy': 'single', 'provider': provider, 'finder_id': rec.get('id'),
        'answer': resp.text, 'tokens': resp.usage.get('total_tokens', 0),
        'latency_ms': int((time.perf_counter() - t0) * 1000), 'calls': 1,
    }

## 5.2 Strategy B — Self-Reflection

2-pass: draft → critic → revise. Critic이 비판할 게 없으면 Pass 2 생략(비용 절감).

In [ ]:
CRITIC_SYS = (
    'You are a strict reviewer. Examine the candidate answer against the context. '
    'List concrete weaknesses: missing evidence, fabricated claims, incorrect citations. '
    'If none, reply with the literal string "NO_ISSUES".'
)
REVISE_SYS = (
    'Revise the previous answer to fix the listed issues. Keep the citation envelope.'
)

def self_reflect(rec: dict, provider: str = 'openai') -> dict:
    sys_p, user = build_prompt(rec)
    t0 = time.perf_counter()
    draft = chat(provider, user=user, system=sys_p, max_tokens=300, temperature=0.0)
    critic = chat(
        provider,
        user=f'Question: {rec.get("question")}\nContext:\n{rec.get("text", "")[:1600]}\n\nCandidate answer:\n{draft.text}',
        system=CRITIC_SYS, max_tokens=250, temperature=0.0,
    )
    tok_total = draft.usage.get('total_tokens', 0) + critic.usage.get('total_tokens', 0)
    calls = 2
    final_text = draft.text
    if 'NO_ISSUES' not in critic.text:
        rev = chat(
            provider,
            user=f'Issues:\n{critic.text}\n\nOriginal answer:\n{draft.text}\n\nProduce the revised answer only.',
            system=REVISE_SYS, max_tokens=300, temperature=0.0,
        )
        final_text = rev.text
        tok_total += rev.usage.get('total_tokens', 0)
        calls = 3
    return {
        'strategy': 'self_reflect', 'provider': provider, 'finder_id': rec.get('id'),
        'answer': final_text, 'tokens': tok_total,
        'latency_ms': int((time.perf_counter() - t0) * 1000), 'calls': calls,
    }

## 5.3 Strategy C — Multi-LLM Debate (4 provider)

1. 각 provider가 독립적으로 draft
2. round 1: 다른 provider 의 draft에 반박/보완
3. moderator (OpenAI) 가 합성 + citation 통합

In [ ]:
DEBATE_SYS = (
    'You are a debate participant. Critique the other participants\' answers, '
    'point out missing evidence, and improve your own answer based on the discussion. '
    'Be concise. Cite sources as [src:id]. Output the new answer only.'
)
MODERATOR_SYS = (
    'You are the debate moderator. Synthesize the participants\' answers into a single final '
    'answer. Preserve citations. If participants disagree on a fact, state the disagreement '
    'explicitly. End with <confidence>high|medium|low</confidence>.'
)

def debate(rec: dict, participants: list[str], moderator: str = 'openai') -> dict:
    sys_p, user = build_prompt(rec)
    t0 = time.perf_counter()
    calls = 0
    tok = 0

    # round 0: drafts
    drafts = {}
    for p in participants:
        r = chat(p, user=user, system=sys_p, max_tokens=250, temperature=0.0)
        drafts[p] = r.text; tok += r.usage.get('total_tokens', 0); calls += 1

    # round 1: cross-critique
    revised = {}
    for p in participants:
        others = '\n\n'.join(f'[{q}] {t}' for q, t in drafts.items() if q != p)
        r = chat(
            p,
            user=f'Question: {rec.get("question")}\nContext: {rec.get("text", "")[:1500]}\n\nYour draft:\n{drafts[p]}\n\nOther participants:\n{others}',
            system=DEBATE_SYS, max_tokens=300, temperature=0.0,
        )
        revised[p] = r.text; tok += r.usage.get('total_tokens', 0); calls += 1

    # moderator
    panel = '\n\n'.join(f'[{p}] {t}' for p, t in revised.items())
    mod = chat(
        moderator,
        user=f'Question: {rec.get("question")}\nContext: {rec.get("text", "")[:1500]}\n\nPanel answers:\n{panel}',
        system=MODERATOR_SYS, max_tokens=350, temperature=0.0,
    )
    tok += mod.usage.get('total_tokens', 0); calls += 1
    return {
        'strategy': 'debate', 'provider': '+'.join(participants), 'finder_id': rec.get('id'),
        'answer': mod.text, 'tokens': tok,
        'latency_ms': int((time.perf_counter() - t0) * 1000), 'calls': calls,
    }

## 5.4 Strategy D — Self-Reflect + Debate (Hybrid)

각 participant가 자기 draft를 self-reflect한 *후* debate에 참여. 발산이 줄고 수렴이 빨라진다.

In [ ]:
def hybrid(rec: dict, participants: list[str], moderator: str = 'openai') -> dict:
    sys_p, user = build_prompt(rec)
    t0 = time.perf_counter()
    calls = 0; tok = 0
    # 1. self-reflected drafts
    sr_drafts = {}
    for p in participants:
        sr = self_reflect(rec, provider=p)
        sr_drafts[p] = sr['answer']
        tok += sr['tokens']; calls += sr['calls']
    # 2. one debate round
    revised = {}
    for p in participants:
        others = '\n\n'.join(f'[{q}] {t}' for q, t in sr_drafts.items() if q != p)
        r = chat(
            p,
            user=f'Question: {rec.get("question")}\nContext: {rec.get("text", "")[:1500]}\n\nYour reflected draft:\n{sr_drafts[p]}\n\nOthers:\n{others}',
            system=DEBATE_SYS, max_tokens=300, temperature=0.0,
        )
        revised[p] = r.text; tok += r.usage.get('total_tokens', 0); calls += 1
    # 3. moderator
    panel = '\n\n'.join(f'[{p}] {t}' for p, t in revised.items())
    mod = chat(
        moderator,
        user=f'Question: {rec.get("question")}\nContext: {rec.get("text", "")[:1500]}\n\nPanel (post self-reflection + debate):\n{panel}',
        system=MODERATOR_SYS, max_tokens=350, temperature=0.0,
    )
    tok += mod.usage.get('total_tokens', 0); calls += 1
    return {
        'strategy': 'hybrid', 'provider': '+'.join(participants), 'finder_id': rec.get('id'),
        'answer': mod.text, 'tokens': tok,
        'latency_ms': int((time.perf_counter() - t0) * 1000), 'calls': calls,
    }

### 📎 Debate Convergence Analysis — `chapter-05-debate-convergence-analysis.md`

본편의 4-strategy 비교가 결과 *수치*에 집중했다면 부속 문서는 debate의 *역학*을 다룬다:
- 수렴(convergence)의 3가지 정의 — 텍스트 유사도 / 인용 set Jaccard / fact 일치
- Early-stop 기준 5종 (수렴 도달, 정체, hard cap, time, cost) + 트리거 추적
- Participant 선정 휴리스틱 (intent → providers)
- Moderator failover chain + self-check + conflict resolution policy
- 5가지 anti-pattern (echo chamber / sycophancy / moderator bias / context drop / citation drift)

부속 문서: [`chapter-05-debate-convergence-analysis.md`](./chapter-05-debate-convergence-analysis.md)

**원칙**: *debate 의 비용은 round 수에 비례하고, 정확도 개선은 체감한다. 수렴을 정의하고, 측정하고, 멈춰라.*

In [ ]:
# 부속 문서 §1~§2: 수렴 곡선 (citation Jaccard) + early-stop 판정
import re
from statistics import mean

CITE_RE_2 = re.compile(r'\[src:([^\]\s,]+)(?:,\s*chunk:(\d+))?\]')

def extract_citations(text: str) -> set[tuple]:
    return {(m[0], m[1] or '') for m in CITE_RE_2.findall(text)}

def convergence_curve(per_round_panels: list[dict]) -> list[float]:
    """각 round 의 panel 답변쌍의 citation Jaccard 평균."""
    curve = []
    for panel in per_round_panels:
        cite_sets = [extract_citations(a) for a in panel.values()]
        pairs = [
            len(a & b) / max(1, len(a | b))
            for i, a in enumerate(cite_sets)
            for j, b in enumerate(cite_sets) if j > i
        ]
        curve.append(mean(pairs) if pairs else 0.0)
    return curve

def should_stop(curve: list[float], elapsed_ms: int, tokens: int, max_rounds: int = 3) -> tuple[bool, str]:
    if not curve: return False, ''
    if curve[-1] >= 0.80: return True, f'convergence reached ({curve[-1]:.2f})'
    if len(curve) >= 3 and abs(curve[-1]-curve[-2]) < 0.05 and abs(curve[-2]-curve[-3]) < 0.05:
        return True, 'no improvement (2 stagnant rounds)'
    if len(curve) >= max_rounds: return True, f'hard cap = {max_rounds}'
    if elapsed_ms >= 60_000: return True, 'time budget'
    if tokens >= 30_000: return True, 'token budget'
    return False, ''

# 데모: 가상의 3 라운드 panel 답변에서 수렴 곡선을 그려본다
rounds = [
    {'A': 'cyber risk [src:f1]', 'B': 'supply chain [src:f2]', 'C': 'ransomware [src:f3]'},
    {'A': 'cyber + supply chain [src:f1][src:f2]', 'B': 'supply chain + cyber [src:f1][src:f2]', 'C': 'cyber risk [src:f1]'},
    {'A': 'cyber + supply [src:f1][src:f2]', 'B': 'cyber + supply [src:f1][src:f2]', 'C': 'cyber + supply [src:f1][src:f2]'},
]
curve = convergence_curve(rounds)
print('convergence curve (round Jaccard):', [round(x, 3) for x in curve])
stop, why = should_stop(curve, elapsed_ms=12_000, tokens=8_000)
print(f'early-stop? {stop}  reason: {why or "continue"}')

## 5.5 4 Strategy 비교 — 같은 평가셋

In [ ]:
import pandas as pd
LIVE = os.getenv('CH05_LIVE', '0') == '1' and len(CONFIGURED) >= 2
if not LIVE:
    print('CH05_LIVE=1 + 최소 2 provider 설정 시 4-strategy 비교 실행.')
else:
    # 비용 절감을 위해 평가셋 일부만 사용
    SAMPLE = EVAL[:3]
    participants = CONFIGURED[:3] if len(CONFIGURED) >= 3 else CONFIGURED
    rows = []
    for rec in SAMPLE:
        rows.append(single(rec, provider=CONFIGURED[0]))
        rows.append(self_reflect(rec, provider=CONFIGURED[0]))
        rows.append(debate(rec, participants=participants, moderator=CONFIGURED[0]))
        rows.append(hybrid(rec, participants=participants, moderator=CONFIGURED[0]))
    out = pd.DataFrame(rows)
    print('per-strategy averages:')
    print(out.groupby('strategy').agg(
        tokens=('tokens', 'mean'),
        latency_ms=('latency_ms', 'mean'),
        calls=('calls', 'mean'),
    ).round(1))
    print()
    display(out[['strategy', 'finder_id', 'tokens', 'latency_ms', 'calls']])

## 5.6 환각률 / 인용 유효성 자동 측정

In [ ]:
import re
CITE_RE = re.compile(r'\[src:([^\]\s]+)\]')

def hallucination_rate(answer: str, valid_ids: set[str]) -> float:
    cites = CITE_RE.findall(answer)
    if not cites: return 0.0  # citation 자체가 없으면 별도 신호로
    invalid = [c for c in cites if c not in valid_ids]
    return len(invalid) / len(cites)

if LIVE:
    valid = {rec['id'] for rec in SAMPLE}
    out['hallucination_rate'] = out['answer'].apply(lambda a: hallucination_rate(a, valid))
    print(out.groupby('strategy')['hallucination_rate'].mean().round(3))

**기대 결과**

| Strategy | 정확도 | 환각률 | 토큰(상대) | Latency(상대) |
|---|---|---|---|---|
| Single | baseline | baseline | 1.0× | 1.0× |
| Self-reflect | ↑ | ↓↓ | 2.0× | 2.0× |
| Debate | ↑↑ | ↓ | 3~5× | 3× |
| Hybrid | ↑↑↑ | ↓↓ | 4~6× | 3~4× |

Opik에서 봐야 할 것
- reasoning step 수 vs 정확도 산점도
- 모델별 token 비용 분포
- debate round 별 panel answer 유사도 변화 (수렴 곡선)

In [ ]:
teardown_opik()
print('chapter 05 done →', trace_info.get('jsonl_path'))
if trace_info.get('project'):
    print('Opik:', opik_console_link(trace_info['project']))

## 시리즈 마무리

- Ch 1 인덱싱 → Ch 2 품질 진단 → Ch 3 자연어→Cypher → Ch 4 라우팅 → Ch 5 debate
- 모든 챕터에서 **동일한 4-provider 인터페이스**와 **Opik per-user 프로젝트**로 trace 비교 가능
- FinDER 같은 데이터셋을 카테고리별/랜덤/N-per-category로 자유롭게 샘플링 가능

후속 작업 후보
- 실제 벤치마크 (BenchmarkRunner) 와 통합해 정답 매칭 metric 추가
- 카테고리별 win-rate 시각화
- 비용 분석 dashboard (Opik metadata + parquet 누적)